In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)

## Phase 1 Architecture

```
PDFs → cleaner → chunker → embedder → pgvector
         ↓ (query time)
Query → embed → cosine search → [optional: BM25 + RRF] → context → LLM → answer
```

**What we built:**
- 8-step text cleaning pipeline
- 4 chunking strategies (RECURSIVE chosen)
- OpenAI + ONNX embedders
- pgvector with HNSW index
- First RAG chain (LCEL)
- Hybrid BM25+vector retrieval
- Metadata filtering + self-querying

In [ ]:
from rag.retrieval.vector_retriever import VectorRetriever
from rag.chains.rag_chain import invoke
ret = VectorRetriever()
test_qs = [
    'What is the total fertility rate in Nigeria?',
    'How does contraceptive use vary between urban and rural women in Ghana?',
    'What is the under-5 mortality rate in Kenya?',
    'Compare maternal health outcomes between Nigeria and Kenya.',
]
results = []
for q in test_qs:
    docs = ret.retrieve(q, top_k=5)
    ans  = invoke(q, ret)
    results.append({'question':q,'answer':ans,'n_docs':len(docs)})
    print(f'Q: {q[:70]}')
    print(f'A: {ans[:200]}')
    print(f'Docs: {len(docs)}\n')

In [ ]:
failure_qs = [
    'What is the maternal mortality ratio in Nigeria?',  # may not be in mini-report
    'DHS 2022 FR380 volume II fertility preferences',  # specific file reference
    'What does the erratum say was corrected?',  # FR380erratum test
]
print('Testing failure modes...')
for q in failure_qs:
    docs = ret.retrieve(q, top_k=3)
    ans  = invoke(q, ret)
    sources = [d.metadata.get('file_name','?') for d in docs]
    print(f'Q: {q}')
    print(f'Sources: {sources}')
    print(f'A: {ans[:150]}')
    print()

In [ ]:
print('Phase 1 known limitations:')
print()
print('1. NO RERANKING   → top-k by cosine alone misses best chunks')
print('2. NO EVALUATION  → we have no score to improve against (fixed in E09)')
print('3. NO QUERY REWRITE → user queries are bad retrieval queries (E11)')
print('4. NO CITATIONS    → responses reference [Source N] but no Pydantic schema (E12)')
print('5. NO MEMORY       → each question independent, no conversation (E14)')
print()
print('Phase 2 (Episodes 9-17) addresses all five.')
print('Episode 9: RAGAS evaluation — this is where we establish the baseline score.')

## ✅ Phase 1 complete

**Episode 9:** RAGAS evaluation — you can't improve what you don't measure.
We establish baseline faithfulness, answer relevance, context precision, context recall.